#API Example (yfinance
---
Topics

*   1. Introduction to yfinance API
*   2. Loading data
*   3. YFinance examples
*   4. YFinance Template for aggregating data



#1. Introduction to yfinance API

In [ ]:
#Loading and installing package
!pip install yfinance

In [ ]:
#Check if package is loaded
!pip show yfinance

Name: yfinance
Version: 0.2.66
Summary: Download market data from Yahoo! Finance API
Home-page: https://github.com/ranaroussi/yfinance
Author: Ran Aroussi
Author-email: ran@aroussi.com
License: Apache
Location: /usr/local/lib/python3.12/dist-packages
Requires: beautifulsoup4, curl_cffi, frozendict, multitasking, numpy, pandas, peewee, platformdirs, protobuf, pytz, requests, websockets
Required-by: 


yfinance also has a few dependencies:

**pandas**:  
Converts downloaded financial data into tables (DataFrames) so we can analyze prices, returns, and volume.

**numpy**:  
Used for numerical operations and handling arrays efficiently.

**requests**:  
Sends HTTP requests to Yahoo Finance servers to fetch stock, ETF, and market data.

**lxml**:  
Parses HTML/XML when extracting structured data from Yahoo Finance pages.

**pytz**:  
Handles time zones correctly for datetime indices in historical data.

**datetime (Python stdlib)**:  
Used to manage start and end dates for downloading historical data.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import lxml
import pytz
import datetime

##The following yfinance API methods will be used for data collection, feature engineering, and forecasting.


### Stock Price & Market Data (for Forecasting)
- `yf.download()` – Historical stock, ETF, or index prices (daily, weekly, monthly) used for time series forecasting.
- `Ticker.history()` – Historical OHLCV data for a single ticker (alternative to `yf.download()`).
- `Ticker.info['currentPrice']` – Real-time price (optional) for model evaluation or plotting.


#2. Loading data

## How to download historical data using the Yahoo Finance API

Historical price data is the one thing we will probably almost always need.

The method to get this in the `yfinance` library is `yf.download()` or `Ticker.history()`. We will have to import it from `yfinance`, so we do:



In [ ]:
# Download daily Amazon data from 2010 to 2020
amazon_daily = yf.download(
    tickers="AMZN",
    start="2010-01-01",
    end="2020-01-01",
    interval="1d",
    auto_adjust=True
)

# Display first rows
amazon_daily.head()


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AMZN,AMZN,AMZN,AMZN,AMZN
Date,,,,,
2010-01-04,6.6950,6.8305,6.6570,6.8125,151998000
2010-01-05,6.7345,6.7740,6.5905,6.6715,177038000
2010-01-06,6.6125,6.7365,6.5825,6.7300,143576000
2010-01-07,6.5000,6.6160,6.4400,6.6005,220604000
2010-01-08,6.6760,6.6840,6.4515,6.5280,196610000


Columns: Open, High, Low, Close, Volume

**We then pass in the following arguments to our function:**

- `tickers`: case-insensitive ticker symbol of the desired stock, ETF, or index (e.g., `"AMZN"`).

- `start`: start date of the data in `"YYYY-MM-DD"` format. Example: `"2009-12-04"`.

- `end`: end date of the data in `"YYYY-MM-DD"` format. Example: `"2019-12-04"`.

- `interval`: {`"1d"`, `"1wk"`, `"1mo"`}. Refers to the sampling interval of the data:
  - `"1d"` = daily
  - `"1wk"` = weekly
  - `"1mo"` = monthly

- `group_by`: {`"ticker"`, `"column"`}. Default is `"column"`. Determines how multiple tickers are grouped in the resulting DataFrame.

- `auto_adjust`: {True, False}. Default is True. If True, automatically adjusts OHLC for splits and dividends.

- `prepost`: {True, False}. Default is False. If True, includes pre-market and after-hours data for daily interval.

- `threads`: Number of threads to use for downloading multiple tickers. Default is True (auto).

#3. yfinance examples

Weekly data is often better for medium- to long-term forecasting:

In [ ]:
# Download weekly data
apple_weekly = yf.download(
    tickers="AAPL",
    start="2025-01-01",
    end="2026-01-01",
    interval="1wk",
    auto_adjust=True
)

apple_weekly.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2025-01-01,240.894073,247.746639,240.038745,247.577550,181886400
2025-01-08,232.012573,242.385914,228.471917,240.605631,188405800
2025-01-15,221.430374,237.661713,218.188092,233.365177,278149800
2025-01-22,236.965530,238.885053,218.595877,218.595877,349630200
2025-01-29,231.535202,245.847021,224.473770,232.848023,320234800


You can download multiple tickers at once:

In [ ]:
tickers = ["AMZN", "AAPL", "MSFT"]

data = yf.download(
    tickers=tickers,
    start="2010-01-01",
    end="2020-01-01",
    interval="1mo",  # Monthly data
    auto_adjust=True
)

data.head()


[*********************100%***********************]  3 of 3 completed


Price          Close                         High                     \
Ticker          AAPL    AMZN       MSFT      AAPL    AMZN       MSFT   
Date                                                                   
2010-01-01  5.754695  6.2705  21.011976  6.459725  6.8305  23.293617   
2010-02-01  6.131031  5.9200  21.377338  6.147511  6.2430  21.645767   
2010-03-01  7.041305  6.7885  21.941755  7.115614  6.9095  22.900629   
2010-04-01  7.823043  6.8550  22.878155  8.163722  7.5545  23.657240   
2010-05-01  7.696898  6.2730  19.327322  8.026491  6.9720  23.267699   

Price            Low                         Open                     \
Ticker          AAPL    AMZN       MSFT      AAPL    AMZN       MSFT   
Date                                                                   
2010-01-01  5.700462  5.9060  20.624246  6.395005  6.8125  22.831324   
2010-02-01  5.718440  5.6910  20.557140  5.763984  6.1590  21.168560   
2010-03-01  6.155898  5.8765  21.155177  6.164887  5.9350  21.552212   
2010-04-01  6.973891  6.5390  21.439842  7.113519  6.7900  21.986700   
2010-05-01  5.970129  5.8760  18.398412  7.905441  6.8600  22.975542   

Price            Volume                          
Ticker             AAPL        AMZN        MSFT  
Date                                             
2010-01-01  15168994400  4617220000  1359650900  
2010-02-01  10776080000  4202916000  1074643300  
2010-03-01  12154172800  3160852000  1110237200  
2010-04-01  12367129600  3460502000  1319029500  
2010-05-01  18082654800  2818198000  1720130200

Forecasting models usually work with a single time series column. For example, we can select just the Close prices:

In [ ]:
# Daily closing prices
ts_daily = amazon_daily['Close']

# Weekly closing prices




Ticker,AMZN
Date,
2010-01-01,6.5000
2010-01-08,6.3675
2010-01-15,6.3310
2010-01-22,6.3015
2010-01-29,5.7970


#4. YFinance Template for aggregating data

In [ ]:
# Inputs
TICKER = ""               # Stock/ETF/index ticker
START_DATE = "YEAR-MM-DD"     # Start date (YYYY-MM-DD)
END_DATE = "YEAR-MM-DD"       # End date (YYYY-MM-DD)
INTERVAL = ""              # Options: "1d", "1wk", "1mo"
AUTO_ADJUST = True            # Adjust OHLC for splits/dividends

raise NotImplementedError #Delete this after filling out parameters

# Download Historical Data
data = yf.download(
    tickers=TICKER,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=AUTO_ADJUST
)

IndexError: string index out of range